In [0]:
%sql
-- Create catalog
CREATE CATALOG test_demo COMMENT 'Demo catalog for lakeflow medallion architecture';

-- Create schemas
CREATE SCHEMA test_demo.bronze COMMENT 'Bronze layer schema for raw data';
CREATE SCHEMA test_demo.silver COMMENT 'Silver layer schema for cleaned/validated data';
CREATE SCHEMA test_demo.gold COMMENT 'Gold layer schema for curated star schema tables';

-- Create managed volume for bronze landing data
CREATE VOLUME test_demo.bronze.landing COMMENT 'Landing volume for bronze raw files';

GRANT ALL PRIVILEGES ON CATALOG test_demo TO `miguel.palomino@uao.edu.co`;


In [0]:
SHOW CATALOGS;

In [0]:
# Configurar catálogo y schema
spark.sql("USE CATALOG test_demo")
spark.sql("USE SCHEMA bronze")

# Leer el archivo CSV
csv_path = "/Volumes/test_demo/bronze/landing/fx_precios_ultimo_anio_VE_AR_MX_BR.csv"  # Ajusta la ruta

# Opción A: Con inferSchema automático
df_csv = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .csv(csv_path)
"""
schema = StructType([
    StructField("country", StringType(), True),
    StructField("date", StringType(), True),
    StructField("exchange_rate", DoubleType(), True),
    StructField("currency", StringType(), True)
    # Agrega más campos según tu CSV
])

df_csv = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(csv_path)
"""

# Agregar metadata
from pyspark.sql.functions import current_timestamp, lit

df_csv_bronze = df_csv.withColumn("extraction_timestamp", current_timestamp()) \
                      .withColumn("source_type", lit("CSV")) \
                      .withColumn("processing_date", current_date())

# Guardar en el catálogo
df_csv_bronze.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("test_demo.bronze.historical_rates_csv")

print("✅ CSV cargado en test_demo.bronze.historical_rates_csv")

✅ CSV cargado en test_demo.bronze.historical_rates_csv


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("Moneda", StringType(), True),
    StructField("Pais", StringType(), True),
    StructField("Fecha", TimestampType(), True),
    StructField("Compra", DoubleType(), True),
    StructField("Venta", StringType(), True)
])

from pyspark.sql.functions import (
    current_timestamp, lit, current_date, upper, col, when
)

csv_path = "/Volumes/test_demo/bronze/landing/fx_precios_ultimo_anio_VE_AR_MX_BR.csv"

df_csv = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .option("dateFormat", "yyyy-MM-dd HH:mm:ss") \
    .csv(csv_path).withColumn('Fecha', col('Fecha').cast(TimestampType()))

df_csv_bronze = df_csv \
    .withColumn("Moneda", upper(col("Moneda"))) \
    .withColumn("Moneda", when(col("Moneda") == "DOLLAR", "USD").otherwise(col("Moneda"))) \
    .withColumn("Venta", when(col("Venta") == 0, None).otherwise(col("Venta"))) \
    .withColumn("extraction_timestamp", current_timestamp()) \
    .withColumn("source_type", lit("CSV_HISTORICAL")) \
    .withColumn(
        "data_quality_flag",
        when(col("Venta").isNull() | (col("Venta") == 0), "MISSING_VENTA")
        .when(col("Compra").isNull() | (col("Compra") == 0), "MISSING_COMPRA")
        .otherwise("OK")
    )

df_csv_bronze.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("test_demo.bronze.historical_rates_csv")

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8111121938978139>, line 39
     17 df_csv = spark.read \
     18     .option("header", "true") \
     19     .schema(schema) \
     20     .option("dateFormat", "yyyy-MM-dd HH:mm:ss") \
     21     .csv(csv_path).withColumn('Fecha', col('Fecha').cast(TimestampType()))
     23 df_csv_bronze = df_csv \
     24     .withColumn("Moneda", upper(col("Moneda"))) \
     25     .withColumn("Moneda", when(col("Moneda") == "DOLLAR", "USD").otherwise(col("Moneda"))) \
   (...)
     33         .otherwise("OK")
     34     )
     36 df_csv_bronze.write \
     37     .mode("overwrite") \
     38     .option("overwriteSchema", "true") \
---> 39     .saveAsTable("test_demo.bronze.historical_rates_csv")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, for